In [1]:
import sys
import os
from scipy.optimize import curve_fit
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.io import shapereader as shpreader
from scipy.integrate import cumulative_trapezoid
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np


# go one folder up from the current working directory
parent_dir = os.path.abspath("real_data")

# build the path to folder_b
path = os.path.join(parent_dir)

# add it to sys.path
sys.path.append(path)


#import vector_borne_functions as vbf
import real_data_functions as real_f
import pandas as pd
import xarray as xr

T = real_f.T
f = real_f.f
gammas = real_f.gammas
g = real_f.g
chis = real_f.chis
F = real_f.F
daily_mean_arrays = real_f.daily_mean_arrays

dX_ms = real_f.dX_ms
Vector_borne_ms = real_f.Vector_borne_ms


plot_compartments_vs_t = real_f.plot_compartments_vs_t

In [2]:
def sinusoid(t,a,b,delta):
    
    return (a+b)/2+(b-a)*(-np.cos(t*2*np.pi+delta))/2

def f_(T):
    return float(f([T])[0])

def g_(T):
    return float(g([T])[0])


def get_sinusoidal_fit(T_t):

    t = np.array(range(len(T_t)))/(365*24)
    # Ajustar usando curve_fit
    popt, pcov = curve_fit(sinusoid, t, T_t, p0=[-5, 50,0.1])  # p0 = estimación inicial [a,b]

    a_fit, b_fit,delta_fit = popt

    return sinusoid(t, a_fit,b_fit,delta_fit), t, a_fit, b_fit


def cumulative_trapz_positive(f_t, g_t, t):
    integrand = f_t - g_t
    I = np.zeros_like(t)

    for i in range(1, len(t)):
        dt = t[i] - t[i-1]
        # regla del trapecio
        increment = 0.5 * (integrand[i] + integrand[i-1]) * dt
        I[i] = max(0.0, I[i-1] + increment)

    return I

def sum_alpha_i(MGDD_R):
    

    c_1 = 0.012
    c_2 = 975
    
    sum_alpha = MGDD_R + 1/c_1*np.log( (1 + np.exp(-c_1 * (MGDD_R-c_2) ) ) /(1 + np.exp(c_1 * c_2) ) ) 

    return sum_alpha


def sum_g(n,mean_g_t_array, mean_f_t_array):

    MGDD_R = 1500.

    Delta_MGDD=MGDD_R/n

    MGDD = np.array([x*Delta_MGDD for x in range(n+1)])

    l_MGDD = len(MGDD)

    s=0
    for i in range(1,n-2):

        s_=0

        for j in range(i-1,n-3):

            s_+= (mean_g_t_array/mean_f_t_array)**(n-2-j)

        s+=F(MGDD[i])*s_

    return s/mean_f_t_array*Delta_MGDD


def R_0(mean_f_t_array,mean_g_t_array):

    MGDD_R = 1500.
    n=200

    alpha_sum  = sum_alpha_i(MGDD_R)
    sum_g_f = sum_g(200,mean_g_t_array, mean_f_t_array)

    R_0 = R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum / (mean_f_t_array)) + sum_g_f)  * (1-(mean_g_t_array / mean_f_t_array))/(1-(mean_g_t_array / mean_f_t_array)**(n-1))
    R_0_aproximation_array [R_0_aproximation_array <0] = 0

    return R_0_aproximation_array


def get_lan_lon(point):

    lat,lon = temperature_dataframe[temperature_dataframe.points == point][['lat','lon']].iloc[0]

    return lat, lon

In [7]:
"""
ds = xr.open_dataset("chelsa_spain_random_points.nc")
ds_apulia = xr.open_dataset("chelsa_apulia.nc")
ds_corcega = xr.open_dataset("chelsa_corsica.nc")

ds = ds.isel(points=[3870, 5709])

lat_apulia = 40.87438066753962
lon_apulia = 16.703334406381828 

lat_corcega = 42.20022914062249
lon_corcega = 8.864759081056688 


new_point1 = int(ds.points.max()) + 1
new_point2 = new_point1 + 1

ds_apulia_exp = ds_apulia.expand_dims(points=[new_point1])
ds_corcega_exp = ds_corcega.expand_dims(points=[new_point2])


ds_apulia_exp["lat"] = ("points", [lat_apulia])
ds_apulia_exp["lon"] = ("points", [lon_apulia])

ds_corcega_exp["lat"] = ("points", [lat_corcega])
ds_corcega_exp["lon"] = ("points", [lon_corcega])

ds = xr.concat(
    [ds, ds_apulia_exp, ds_corcega_exp],
    dim="points"
)

"""

ds = xr.open_dataset("selected_points_.nc")
ds = ds.rename({"station": "points"})

temperature_dataframe = ds.to_dataframe().reset_index()

# df: columns = ["date", "point", "temperature"]
temperature_dataframe_wide = temperature_dataframe.pivot(index="points", columns="time", values="tas")
temperature_dataframe_wide.iloc[:, :] = temperature_dataframe_wide.iloc[:, :] - 273.15


In [8]:
Alpha = 5.
beta = 1.
Gamma = 1.
mu = 1.
delta = mu

df_f_t = temperature_dataframe_wide.map(lambda x: f_(x)*365)

df_g_t = temperature_dataframe_wide.map(lambda x: g_(x)*365)

number_of_columns = df_f_t.shape[1]

times = np.array(range(number_of_columns))/365


cumulated_mgdd = []

for i in range(len(df_f_t)):
    I_row = cumulative_trapz_positive(
        df_f_t.iloc[i].to_numpy(),
        df_g_t.iloc[i].to_numpy(),
        times
    )
    cumulated_mgdd.append(I_row)

cumulated_mgdd = pd.DataFrame(
    cumulated_mgdd,
    index=df_f_t.index,
    columns=df_g_t.columns
)

sinusoidal_fits = temperature_dataframe_wide.apply(get_sinusoidal_fit,axis = 1)


columns_temperature = temperature_dataframe_wide.columns
#sinusoidal_temperatures_profiles = pd.DataFrame(sinusoidal_fits.apply(lambda x: x[0]).tolist(),columns = columns_temperature)

sinusoidal_temperatures_profiles = pd.DataFrame(
    sinusoidal_fits.apply(lambda x: x[0]).tolist(),
    index=temperature_dataframe_wide.index,   
    columns=columns_temperature
)


a_fited = sinusoidal_fits.apply(lambda x: x[2])
b_fited = sinusoidal_fits.apply(lambda x: x[3])


noise_dataframe = temperature_dataframe_wide - sinusoidal_temperatures_profiles
mean_noise  = noise_dataframe.apply(np.mean, axis=1)
var_noise = noise_dataframe.apply(np.var, axis=1)




# DataFrame resumen
df_summary = pd.DataFrame()

# Promedio de cada fila
df_summary['$\\bar{T}$'] = temperature_dataframe_wide.apply(np.mean, axis=1)

# Desviación estándar de cada fila
df_summary['$\\sigma ^ 2_{T}$'] = temperature_dataframe_wide.apply(np.var, axis=1)

bar_f = df_f_t.mean(axis = 1)
bar_g = df_g_t.mean(axis = 1)

df_summary['$\\bar{f}$'] = bar_f

df_summary['$\\frac{\\bar{g}}{\\bar{f}}$'] = bar_g/bar_f

df_summary['$R_0}$'] = R_0(bar_f,bar_g)



df_summary['$a$'] = a_fited

df_summary['$b$'] = b_fited

df_summary['$\\mu_{\\delta}$'] = mean_noise

df_summary['$\\sigma_{\\delta} ^ 2$'] = var_noise


lat_lon_unique_df = temperature_dataframe[['lat','lon']].drop_duplicates()

df_summary['lat'] = lat_lon_unique_df['lat'].values
df_summary['lon'] = lat_lon_unique_df['lon'].values

df_summary.to_csv('df_summary.csv')